# Nuclear Waste Canister Temperature Prediction
**CIVIL-226 - Introduction to Machine Learning for Engineers**  
**Team name:** [À remplir]  
**Members:** [À remplir]

## Objectif
Prédire la température autour de conteneurs de déchets nucléaires à des positions de capteurs non observés, en utilisant la puissance de chauffage, le temps, et les coordonnées spatiales 3D des capteurs.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Imports OK')

## 2. Chargement des données

In [ ]:
sensors = pd.read_parquet('data/sensors.parquet')
train   = pd.read_parquet('data/train.parquet')
test    = pd.read_parquet('data/test.parquet')

print(f'Sensors : {sensors.shape}  ->  {sensors.columns.tolist()}')
print(f'Train   : {train.shape}   ->  {train.columns.tolist()}')
print(f'Test    : {test.shape}    ->  {test.columns.tolist()}')

## 3. Exploration des données (EDA)

In [ ]:
print('=== SENSORS ===')
display(sensors.head())
print(f'\n{sensors["sensor"].nunique()} capteurs uniques')

print('\n=== TRAIN ===')
display(train.head())
print(f'\nValeurs manquantes :')
print(train.isnull().sum())

In [ ]:
# Distribution de la température cible
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(train['temperature'].dropna(), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution des températures')
axes[0].set_xlabel('Température')
axes[0].set_ylabel('Count')

# Évolution temporelle (moyenne sur tous les capteurs)
temp_by_time = train.groupby('time')['temperature'].mean()
axes[1].plot(temp_by_time.index / 1e6, temp_by_time.values, color='tomato', linewidth=0.8)
axes[1].set_title('Température moyenne au cours du temps')
axes[1].set_xlabel('Temps (x10⁶ s)')
axes[1].set_ylabel('Température moyenne')

plt.tight_layout()
plt.show()

In [ ]:
# Positions des capteurs dans l'espace
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# Séparer capteurs train vs test
train_sensors = set(train['sensor'].unique())
test_sensors  = set(test['sensor'].unique())

s_train = sensors[sensors['sensor'].isin(train_sensors)]
s_test  = sensors[sensors['sensor'].isin(test_sensors)]

ax.scatter(s_train['coor_x'], s_train['coor_y'], s_train['coor_z'],
           c='steelblue', label='Train sensors', alpha=0.6, s=20)
ax.scatter(s_test['coor_x'], s_test['coor_y'], s_test['coor_z'],
           c='tomato', label='Test sensors', alpha=0.9, s=40, marker='^')

ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title('Positions des capteurs (3D)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Preprocessing

### 4.1 Gestion des valeurs manquantes
Le train set contient ~99k valeurs de température manquantes. On les supprime car on ne peut pas les utiliser pour entraîner le modèle.

In [ ]:
print(f'Lignes avant nettoyage : {len(train)}')
train_clean = train.dropna(subset=['temperature']).copy()
print(f'Lignes après suppression NaN : {len(train_clean)}')
print(f'Supprimées : {len(train) - len(train_clean)} ({100*(len(train)-len(train_clean))/len(train):.1f}%)')

### 4.2 Feature Engineering

On joint les positions 3D des capteurs au dataframe, et on ajoute des features dérivées utiles.

In [ ]:
def add_features(df, sensors_df):
    """
    Joint les coordonnées 3D et ajoute des features dérivées.
    
    Features ajoutées :
    - coor_x, coor_y, coor_z : position dans l'espace
    - dist_center : distance à l'axe central du canister (r = sqrt(x²+y²))
    - time_norm : temps normalisé entre 0 et 1
    """
    merged = df.merge(sensors_df, on='sensor', how='left')
    
    # Distance radiale au centre du canister
    merged['dist_center'] = np.sqrt(merged['coor_x']**2 + merged['coor_y']**2)
    
    # Temps normalisé
    t_max = train_clean['time'].max()
    merged['time_norm'] = merged['time'] / t_max
    
    return merged

train_feat = add_features(train_clean, sensors)
test_feat  = add_features(test, sensors)

print('Features disponibles :', train_feat.columns.tolist())
display(train_feat.head(3))

In [ ]:
# Features retenues pour le modèle
FEATURES = ['coor_x', 'coor_y', 'coor_z', 'time_norm', 'power', 'dist_center']
TARGET   = 'temperature'

X = train_feat[FEATURES].values
y = train_feat[TARGET].values
X_test = test_feat[FEATURES].values

print(f'X shape : {X.shape}')
print(f'X_test shape : {X_test.shape}')

### 4.3 Split train/validation & Normalisation

In [ ]:
# Split 80/20
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalisation (important pour kNN et Ridge)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

print(f'Train : {X_train_s.shape} | Val : {X_val_s.shape}')

## 5. Modèle Baseline — Ridge Regression

On commence par un modèle simple comme baseline. La régression Ridge est rapide, interprétable, et constitue un bon point de référence.

In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_s, y_train)

y_pred_val_ridge = ridge.predict(X_val_s)
rmse_ridge = np.sqrt(mean_squared_error(y_val, y_pred_val_ridge))

print(f'Baseline Ridge — RMSE validation : {rmse_ridge:.4f}')

## 6. Modèle amélioré — k-Nearest Neighbors

kNN est naturellement adapté à ce problème : la température en un point dépend de ses voisins spatiaux et temporels. On cherche le meilleur k par validation.

In [ ]:
# Recherche du meilleur k (sur un sous-ensemble pour la rapidité)
# On utilise un sample pour éviter des temps de calcul trop longs
sample_size = min(50000, len(X_train_s))
idx = np.random.choice(len(X_train_s), sample_size, replace=False)

k_values = [3, 5, 10, 20]
rmse_k = []

for k in k_values:
    knn = KNeighborsRegressor(n_neighbors=k, n_jobs=-1)
    knn.fit(X_train_s[idx], y_train[idx])
    pred = knn.predict(X_val_s[:5000])  # validation partielle
    rmse = np.sqrt(mean_squared_error(y_val[:5000], pred))
    rmse_k.append(rmse)
    print(f'k={k:3d}  ->  RMSE = {rmse:.4f}')

best_k = k_values[np.argmin(rmse_k)]
print(f'\nMeilleur k : {best_k}')

In [ ]:
# Visualisation de la recherche de k
plt.figure(figsize=(6, 4))
plt.plot(k_values, rmse_k, 'o-', color='steelblue')
plt.axvline(best_k, color='tomato', linestyle='--', label=f'Best k={best_k}')
plt.xlabel('k'); plt.ylabel('RMSE validation')
plt.title('Sélection du hyperparamètre k')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Entraînement final du kNN avec le meilleur k sur toutes les données
knn_final = KNeighborsRegressor(n_neighbors=best_k, n_jobs=-1)
knn_final.fit(X_train_s, y_train)

y_pred_val_knn = knn_final.predict(X_val_s)
rmse_knn = np.sqrt(mean_squared_error(y_val, y_pred_val_knn))
print(f'kNN (k={best_k}) — RMSE validation : {rmse_knn:.4f}')

## 7. Comparaison des modèles

In [ ]:
results = pd.DataFrame({
    'Modèle': ['Ridge Regression (baseline)', f'kNN (k={best_k})'],
    'RMSE Validation': [rmse_ridge, rmse_knn]
})
results = results.sort_values('RMSE Validation')
display(results)

plt.figure(figsize=(6, 3))
plt.barh(results['Modèle'], results['RMSE Validation'], color=['tomato', 'steelblue'])
plt.xlabel('RMSE')
plt.title('Comparaison des modèles')
plt.tight_layout()
plt.show()

## 8. Prédictions finales & Soumission

On utilise le meilleur modèle pour prédire sur le test set.

In [ ]:
# Choisir le meilleur modèle
best_model = knn_final if rmse_knn < rmse_ridge else ridge
print(f'Modèle sélectionné : {"kNN" if best_model is knn_final else "Ridge"}')

# Prédictions sur le test set
y_pred = best_model.predict(X_test_s)

# Créer le fichier de soumission
submission = pd.DataFrame({
    'Id': np.arange(len(test), dtype=int),
    'temperature': y_pred.astype(float)
})

# Vérifications obligatoires
assert list(submission.columns) == ['Id', 'temperature']
assert len(submission) == len(test)
assert (submission['Id'].to_numpy() == np.arange(len(test))).all()
assert np.isfinite(submission['temperature']).all()
assert submission.isna().sum().sum() == 0

submission.to_csv('submission.csv', index=False)
print(f'submission.csv sauvegardé — {len(submission)} lignes')
display(submission.head())

## 9. Pistes d'amélioration

- **Gradient Boosting** (XGBoost, LightGBM) : souvent plus performant que kNN sur ce type de données
- **Réseau de neurones** (PyTorch) : peut capturer des relations non-linéaires complexes
- **Features supplémentaires** : distance au canister en 3D, angle, features temporelles (sin/cos du temps pour capter la périodicité)
- **Interpolation spatiale** : pondérer les voisins par leur distance dans l'espace 3D